# Dropout — Paper-to-Code Mock (Colab)

**Paper:** Dropout: A Simple Way to Prevent Neural Networks from Overfitting (Srivastava et al., 2014) — https://jmlr.org/papers/v15/srivastava14a.html

Warm-up mock (~30 min). Fill in the `Dropout` stub, run the overfitting demo, then the sanity checks. Reference solution in the last cell.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
torch.manual_seed(0)

## 1. Implement inverted dropout (YOUR TASK)

Train: zero each element with prob `p`, scale survivors by `1/(1-p)`. Eval: identity. Respect `self.training`.

In [ ]:
class Dropout(nn.Module):
    def __init__(self, p=0.5):
        super().__init__()
        assert 0.0 <= p < 1.0
        self.p = p

    def forward(self, x):
        # TODO: if not training or p==0 -> return x
        # TODO: sample mask ~ Bernoulli(keep), return x * mask / keep
        raise NotImplementedError("fill me in")

## 2. Demonstrate the benefit (overfitting toy task)
Few noisy training points + oversized MLP. Compare test loss with dropout off vs on.

In [ ]:
in_dim, n_train, n_test = 20, 40, 2000
w_true = torch.randn(in_dim, 1)
Xtr, Xte = torch.randn(n_train, in_dim), torch.randn(n_test, in_dim)
ytr = Xtr @ w_true + 0.5 * torch.randn(n_train, 1)   # noisy targets
yte = Xte @ w_true                                   # clean targets

def make_net(p):
    return nn.Sequential(nn.Linear(in_dim, 256), nn.ReLU(), Dropout(p), nn.Linear(256, 1))

def train_eval(p):
    torch.manual_seed(1)
    net = make_net(p)
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    for _ in range(3000):
        net.train()
        loss = F.mse_loss(net(Xtr), ytr)
        opt.zero_grad(); loss.backward(); opt.step()
    net.eval()
    with torch.no_grad():
        return F.mse_loss(net(Xtr), ytr).item(), F.mse_loss(net(Xte), yte).item()

for p in (0.0, 0.5):
    tr, te = train_eval(p)
    print(f"p={p}:  train {tr:.3f}   test {te:.3f}   gap {te-tr:+.3f}")

## 3. Sanity checks

In [ ]:
# 1) eval is identity
d = Dropout(0.5).eval(); x = torch.randn(1000)
assert torch.equal(d(x), x)

# 2) train drops ~p
d = Dropout(0.3).train(); y = d(torch.ones(100_000))
assert abs((y == 0).float().mean().item() - 0.3) < 0.02

# 3) expectation preserved
d = Dropout(0.5).train()
assert abs(d(torch.full((200_000,), 4.0)).mean().item() - 4.0) < 0.05

# 4) survivors scaled by 1/(1-p)
y = Dropout(0.5).train()(torch.ones(100_000)); nz = y[y != 0]
assert torch.allclose(nz, torch.full_like(nz, 2.0))

# 5) p=0 identity
d = Dropout(0.0).train(); x = torch.randn(1000)
assert torch.equal(d(x), x)

# 6) gradient only through survivors
x = torch.randn(10, requires_grad=True)
Dropout(0.5).train()(x).sum().backward()
assert ((x.grad == 0) | torch.isclose(x.grad, torch.tensor(2.0))).all()

print("All sanity checks passed.")

## 4. Reference solution (peek only after trying)

In [ ]:
class DropoutSolution(nn.Module):
    def __init__(self, p=0.5):
        super().__init__()
        self.p = p

    def forward(self, x):
        if not self.training or self.p == 0.0:
            return x
        keep = 1.0 - self.p
        mask = (torch.rand_like(x) < keep).to(x.dtype)
        return x * mask / keep